In [1]:
import importlib, pathlib, sys
import numpy as np
import pandas as pd
PROJECT_DIR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))
import data, contract, features
importlib.reload(features)

<module 'features' from 'c:\\Users\\Divyanshi\\Desktop\\week3\\Hotel_booking_optimization\\src\\features.py'>

# Hotel cancellation models

Fit on train, select settings on choose, and keep proof untouched until the final evaluation.

In [2]:
raw_df = data.read_raw()
prepared_df = features.prepare_bookings(raw_df)
print("Raw rows:", len(raw_df))
print("Prepared rows:", len(prepared_df))
print("Rows removed by approved filters:", len(raw_df) - len(prepared_df))
print("Missing values after preparation:")
print(prepared_df[["children", "country", "agent", "company"]].isna().sum())

Raw rows: 119390
Prepared rows: 118563
Rows removed by approved filters: 827
Missing values after preparation:
children    0
country     0
agent       0
company     0
dtype: int64


In [3]:
# Official chronological split
train_df, choose_df, proof_df = contract.split_periods(prepared_df)
X_train, y_train, score_train = features.split_model_inputs(train_df)
X_choose, y_choose, score_choose = features.split_model_inputs(choose_df)
X_proof, y_proof, score_proof = features.split_model_inputs(proof_df)
print("Train:", X_train.shape, "Choose:", X_choose.shape, "Proof:", X_proof.shape)
print("Cancellation rates:", round(y_train.mean(), 4), round(y_choose.mean(), 4), round(y_proof.mean(), 4))
print("Index alignment:", X_train.index.equals(y_train.index), X_choose.index.equals(y_choose.index), X_proof.index.equals(y_proof.index))

Train: (48747, 17) Choose: (29307, 17) Proof: (40509, 17)
Cancellation rates: 0.3603 0.3713 0.3883
Index alignment: True True True


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numeric", "passthrough", numeric_features),
])
# Fit only on train; use the fitted transformer for choose and proof.
X_train_processed = preprocessor.fit_transform(X_train)
X_choose_processed = preprocessor.transform(X_choose)
X_proof_processed = preprocessor.transform(X_proof)
print("Processed shapes:", X_train_processed.shape, X_choose_processed.shape, X_proof_processed.shape)

C:\Users\Divyanshi\AppData\Local\Temp\ipykernel_26596\2900665116.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


Processed shapes: (48747, 624) (29307, 624) (40509, 624)


## Baseline model comparison

In [5]:
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

# C=0.01 converged in the earlier check.
logistic_model = LogisticRegression(C=0.01, max_iter=7000, class_weight="balanced", random_state=42)
logistic_model.fit(X_train_processed, y_train)
print("Logistic Regression iterations:", logistic_model.n_iter_[0])
cancel_col = list(logistic_model.classes_).index(1)
choose_prob_lr = logistic_model.predict_proba(X_choose_processed)[:, cancel_col]

Logistic Regression iterations: 3284


In [6]:
from sklearn.tree import DecisionTreeClassifier
decision_tree_model = DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=8, min_samples_leaf=50)
decision_tree_model.fit(X_train_processed, y_train)
cancel_col = list(decision_tree_model.classes_).index(1)
choose_prob_dt = decision_tree_model.predict_proba(X_choose_processed)[:, cancel_col]

In [7]:
from sklearn.ensemble import RandomForestClassifier
random_forest_model = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50, class_weight="balanced", random_state=42, n_jobs=-1)
random_forest_model.fit(X_train_processed, y_train)
cancel_col = list(random_forest_model.classes_).index(1)
choose_prob_rf = random_forest_model.predict_proba(X_choose_processed)[:, cancel_col]

In [8]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=-1)
xgb_model.fit(X_train_processed, y_train)
cancel_col = list(xgb_model.classes_).index(1)
choose_prob_xgb = xgb_model.predict_proba(X_choose_processed)[:, cancel_col]

In [9]:
choose_probabilities = {
    "Logistic Regression (C=0.01)": choose_prob_lr,
    "Decision Tree": choose_prob_dt,
    "Random Forest": choose_prob_rf,
    "XGBoost": choose_prob_xgb,
}
rows = []
for model_name, prob in choose_probabilities.items():
    pred = (prob >= 0.5).astype(int)
    rows.append({"Model": model_name, "ROC-AUC": roc_auc_score(y_choose, prob),
        "Accuracy": accuracy_score(y_choose, pred), "Precision": precision_score(y_choose, pred, zero_division=0),
        "Recall": recall_score(y_choose, pred, zero_division=0), "F1 Score": f1_score(y_choose, pred, zero_division=0),
        "Brier Score": brier_score_loss(y_choose, prob)})
comparison_table = pd.DataFrame(rows).set_index("Model")
display(comparison_table.round(4))

,ROC-AUC,Accuracy,Precision,Recall,F1 Score,Brier Score
Model,,,,,,
Logistic Regression (C=0.01),0.8039,0.7445,0.7043,0.5376,0.6098,0.1691
Decision Tree,0.8231,0.7293,0.6232,0.6850,0.6527,0.1707
Random Forest,0.8092,0.7533,0.8017,0.4459,0.5731,0.1838
XGBoost,0.8428,0.7433,0.8596,0.3690,0.5163,0.1671


## Limited tuning on choose-period nightly cost

Each candidate is fit on train, then its choose probabilities and buffer are scored with the client contract. Proof is not used here.

In [19]:
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
candidates = [
    ("Decision Tree", {"max_depth": 6, "min_samples_leaf": 50}),
    ("Decision Tree", {"max_depth": 8, "min_samples_leaf": 50}),
    ("Decision Tree", {"max_depth": 8, "min_samples_leaf": 100}),
    ("Decision Tree", {"max_depth": 12, "min_samples_leaf": 100}),
    ("XGBoost", {"max_depth": 3, "learning_rate": 0.05, "n_estimators": 200}),
    ("XGBoost", {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 200}),
    ("XGBoost", {"max_depth": 3, "learning_rate": 0.1, "n_estimators": 300}),
    ("XGBoost", {"max_depth": 4, "learning_rate": 0.1, "n_estimators": 300}),
]
results = []
for model_name, params in candidates:
    if model_name == "Decision Tree":
        model = DecisionTreeClassifier(**params, class_weight="balanced", random_state=42)
    else:
        model = XGBClassifier(**params, objective="binary:logistic", eval_metric="logloss", random_state=42, n_jobs=-1)
    model.fit(X_train_processed, y_train)
    cancel_col = list(model.classes_).index(1)
    prob = model.predict_proba(X_choose_processed)[:, cancel_col]
    nights = contract.nightly_table(score_choose, prob)
    max_buffer = int(np.ceil(nights["expected"].max())) + 1
    for buffer in range(max_buffer + 1):
        oversell = np.floor(nights["expected"] - buffer).clip(lower=0).astype(int)
        score = contract.night_cost(nights, oversell)
        results.append({"Model": model_name, "Parameters": str(params), "Buffer": buffer,
            "Cost (€)": score["cost_eur"], "Walks": score["walked_guests"],
            "Walks per 1,000": score["walks_per_1000"], "Ceiling met": score["ceiling_met"]})
tuning_table = pd.DataFrame(results)
acceptable = tuning_table[tuning_table["Ceiling met"]].sort_values("Cost (€)")
display(acceptable.head(10))

,Model,Parameters,Buffer,Cost (€),Walks,"Walks per 1,000",Ceiling met
1393,XGBoost,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",0,1065008.27,184,9.99,True
1169,XGBoost,"{'max_depth': 4, 'learning_rate': 0.05, 'n_est...",1,1182219.37,152,8.25,True
1394,XGBoost,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",1,1184339.48,135,7.33,True
945,XGBoost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",2,1245179.18,145,7.87,True
1623,XGBoost,"{'max_depth': 4, 'learning_rate': 0.1, 'n_esti...",1,1308818.52,165,8.95,True
1170,XGBoost,"{'max_depth': 4, 'learning_rate': 0.05, 'n_est...",2,1311272.10,119,6.46,True
1395,XGBoost,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",2,1313699.74,103,5.59,True
946,XGBoost,"{'max_depth': 3, 'learning_rate': 0.05, 'n_est...",3,1374481.97,116,6.30,True
1624,XGBoost,"{'max_depth': 4, 'learning_rate': 0.1, 'n_esti...",2,1440737.09,137,7.44,True
1171,XGBoost,"{'max_depth': 4, 'learning_rate': 0.05, 'n_est...",3,1441782.97,93,5.05,True


In [20]:
# Candidate safety buffers to compare
buffers = [0, 1, 2, 3, 5, 8, 10]
baseline_results = []

# Flat rule: each hotel's cancellation rate from TRAIN
hotel_rate = score_train.groupby("hotel")["is_canceled"].mean()
prob_flat = (
    X_choose["hotel"]
    .map(hotel_rate)
    .fillna(y_train.mean())
    .to_numpy()
)

# Lookup rule: cancellation rate by hotel + market segment from TRAIN
lookup_train = pd.DataFrame({
    "hotel": X_train["hotel"].to_numpy(),
    "market_segment": X_train["market_segment"].fillna("unknown").astype(str).to_numpy(),
    "cancelled": y_train.to_numpy(),
})

segment_rate = lookup_train.groupby(
    ["hotel", "market_segment"]
)["cancelled"].mean()
hotel_fallback = lookup_train.groupby("hotel")["cancelled"].mean()

choose_segments = X_choose["market_segment"].fillna("unknown").astype(str)
choose_keys = pd.MultiIndex.from_arrays([
    X_choose["hotel"].to_numpy(),
    choose_segments.to_numpy(),
])

prob_lookup = pd.Series(
    segment_rate.reindex(choose_keys).to_numpy(),
    index=X_choose.index,
)
prob_lookup = (
    prob_lookup
    .fillna(X_choose["hotel"].map(hotel_fallback))
    .fillna(y_train.mean())
    .to_numpy()
)

# Score a probability rule and choose its buffer on CHOOSE
def score_buffers(method_name, probabilities):
    nights = contract.nightly_table(score_choose, probabilities)
    rows = []

    max_buffer = int(np.ceil(nights["expected"].max())) + 1
    for buffer in range(max_buffer + 1):
        oversell = np.floor(nights["expected"] - buffer).clip(lower=0).astype(int)
        score = contract.night_cost(nights, oversell)

        rows.append({
            "Model": method_name,
            "Parameters": "hotel cancellation rate" if method_name == "Flat rule"
                          else "hotel + market_segment lookup",
            "Buffer": buffer,
            "Cost (€)": score["cost_eur"],
            "Walks": score["walked_guests"],
            "Walks per 1,000": score["walks_per_1000"],
            "Ceiling met": score["ceiling_met"],
        })

    return rows

baseline_results += score_buffers("Flat rule", prob_flat)
baseline_results += score_buffers("Lookup", prob_lookup)

# Best ceiling-compliant tuned result for each learned method
best_learned = (
    tuning_table[tuning_table["Ceiling met"]]
    .sort_values("Cost (€)")
    .groupby("Model", as_index=False)
    .first()
)

# Combine and show only methods that meet the ceiling
comparison = pd.concat(
    [pd.DataFrame(baseline_results), best_learned],
    ignore_index=True,
)
comparison = comparison[comparison["Ceiling met"]].sort_values("Cost (€)")

flat_cost = comparison.loc[comparison["Model"] == "Flat rule", "Cost (€)"].iloc[0]
comparison["Saving vs flat (€)"] = flat_cost - comparison["Cost (€)"]

display(comparison.reset_index(drop=True).round(2))

,Model,Parameters,Buffer,Cost (€),Walks,"Walks per 1,000",Ceiling met,Saving vs flat (€)
0,XGBoost,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",0,1065008.27,184,9.99,True,1365465.89
1,Decision Tree,"{'max_depth': 12, 'min_samples_leaf': 100}",12,1631450.23,160,8.68,True,799023.93
2,Flat rule,hotel cancellation rate,16,2430474.16,174,9.44,True,0.00
3,Flat rule,hotel cancellation rate,17,2494601.57,153,8.30,True,-64127.41
4,Flat rule,hotel cancellation rate,18,2555121.35,133,7.22,True,-124647.19
...,...,...,...,...,...,...,...,...
278,Lookup,hotel + market_segment lookup,174,4177750.79,0,0.00,True,-1747276.63
279,Lookup,hotel + market_segment lookup,175,4177750.79,0,0.00,True,-1747276.63
280,Lookup,hotel + market_segment lookup,176,4177750.79,0,0.00,True,-1747276.63
281,Flat rule,hotel cancellation rate,134,4177750.79,0,0.00,True,-1747276.63


In [21]:
# Best ceiling-compliant baseline row for each baseline method
baseline_df = pd.DataFrame(baseline_results)
baseline_df = baseline_df[baseline_df["Ceiling met"]]

best_baselines = (
    baseline_df
    .sort_values("Cost (€)")
    .groupby("Model", as_index=False)
    .first()
)

# Best ceiling-compliant tuned row for each learned method
learned_df = tuning_table[tuning_table["Ceiling met"]]

best_learned = (
    learned_df
    .sort_values("Cost (€)")
    .groupby("Model", as_index=False)
    .first()
)

# Combine, calculate savings, and display one row per method
comparison = pd.concat(
    [best_baselines, best_learned],
    ignore_index=True,
).sort_values("Cost (€)")

flat_cost = comparison.loc[
    comparison["Model"] == "Flat rule", "Cost (€)"
].iloc[0]

comparison["Saving vs flat (€)"] = flat_cost - comparison["Cost (€)"]

pd.set_option("display.max_colwidth", None)
display(comparison.reset_index(drop=True).round(2))

,Model,Parameters,Buffer,Cost (€),Walks,"Walks per 1,000",Ceiling met,Saving vs flat (€)
0,XGBoost,"{'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 300}",0,1065008.27,184,9.99,True,1365465.89
1,Decision Tree,"{'max_depth': 12, 'min_samples_leaf': 100}",12,1631450.23,160,8.68,True,799023.93
2,Flat rule,hotel cancellation rate,16,2430474.16,174,9.44,True,0.00
3,Lookup,hotel + market_segment lookup,17,2821440.92,181,9.82,True,-390966.76


In [22]:
from xgboost import XGBClassifier

# Frozen XGBoost choice from the choose-period table
final_xgb = XGBClassifier(
    max_depth=3,
    learning_rate=0.1,
    n_estimators=300,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
final_xgb.fit(X_train_processed, y_train)

cancel_col = list(final_xgb.classes_).index(1)
prob_xgb_proof = final_xgb.predict_proba(X_proof_processed)[:, cancel_col]

# Client flat rule: hotel cancellation rates learned from train
hotel_rate = score_train.groupby("hotel")["is_canceled"].mean()
prob_flat_proof = (
    score_proof["hotel"].map(hotel_rate).fillna(y_train.mean()).to_numpy()
)

def proof_score(method, probabilities, buffer):
    nights = contract.nightly_table(score_proof, probabilities)
    k = np.floor(nights["expected"] - buffer).clip(lower=0).astype(int)
    result = contract.night_cost(nights, k)

    return {
        "Method": method,
        "Buffer": buffer,
        "Cost (€)": result["cost_eur"],
        "Empty rooms": result["empty_rooms"],
        "Walks": result["walked_guests"],
        "Walks per 1,000": result["walks_per_1000"],
        "Ceiling met": result["ceiling_met"],
    }

proof_table = pd.DataFrame([
    proof_score("XGBoost", prob_xgb_proof, buffer=0),
    proof_score("Flat rule", prob_flat_proof, buffer=16),
])

flat_proof_cost = proof_table.loc[
    proof_table["Method"] == "Flat rule", "Cost (€)"
].iloc[0]
proof_table["Saving vs flat (€)"] = flat_proof_cost - proof_table["Cost (€)"]

display(proof_table.round(2))

,Method,Buffer,Cost (€),Empty rooms,Walks,"Walks per 1,000",Ceiling met,Saving vs flat (€)
0,XGBoost,0,1896197.00,3660,201,8.11,True,2032938.35
1,Flat rule,16,3929135.35,8141,154,6.21,True,0.00
